# DSCI 552 Homework 3

**Course:** DSCI 552, Machine Learning for Data Science (Fall 2026)

**Assignment:** Homework 3

**Name:** Zhonghao Yang

**Github Username:** ZhonghaoYang-EDDIE

**Date:** October 12, 2026

**AI Assistance Disclosure:** AI assistance was used (Claude.ai from Anthropic) in this assignment, mainly with code and syntax questions, and for researching common time-domain features in 1(c)i.

In [1]:
import os
import glob
import numpy as np
import pandas as pd

DATA_DIR = os.path.join(os.getcwd(), 'AReM')
print(DATA_DIR, os.path.isdir(DATA_DIR))

ACTIVITIES = ['bending1', 'bending2', 'cycling', 'lying',
              'sitting', 'standing', 'walking']
SERIES = ['avg_rss12', 'var_rss12', 'avg_rss13',
          'var_rss13', 'avg_rss23', 'var_rss23']

c:\Users\15699\OneDrive\桌面\USC Viterbi ADS\DSCI 552\hw 3\AReM True


### Data Cleaning Notes

After inspecting all 88 files, I found that 4 of them had minor formatting problems:

- `bending2/dataset4.csv`: space-delimited instead of comma-delimited, with trailing spaces.
- `cycling/dataset9.csv` and `cycling/dataset14.csv`: the last row ends with an extra comma.
- `sitting/dataset8.csv`: only 479 rows; the row at timestamp 13500 is missing.

I fixed the first three files manually in VS Code (replacing the spaces with commas in `dataset4.csv`, and deleting the extra comma in the two `cycling` files). 

I did not modify `sitting/dataset8.csv` since a single missing row out of 480 has a small effect on summary statistics such as min, max, and mean, so I used its 479 rows as they are.

In [2]:
## 1(b)
def read_instance(path):
    #Read one AReM file and return its 6 time series.
    return pd.read_csv(path, comment='#', header=None, names=['time'] + SERIES)[SERIES]

In [3]:
records = []
for act in ACTIVITIES:
    n_files = len(glob.glob(os.path.join(DATA_DIR, act, 'dataset*.csv')))
    n_test = 2 if act.startswith('bending') else 3   # split rule in 1(b): test set: first 2 files for bending, first 3 for others
    for idx in range(1, n_files + 1):
        path = os.path.join(DATA_DIR, act, f'dataset{idx}.csv')
        records.append({'activity': act, 'dataset': idx,
                        'split': 'test' if idx <= n_test else 'train',
                        'data': read_instance(path)})

meta = pd.DataFrame(records)[['activity', 'dataset', 'split']]
print(len(records), 'instances')
print(pd.crosstab(meta['activity'], meta['split']))
print(pd.Series([r['data'].shape for r in records]).value_counts())

88 instances
split     test  train
activity             
bending1     2      5
bending2     2      4
cycling      3     12
lying        3     12
sitting      3     12
standing     3     12
walking      3     12
(480, 6)    87
(479, 6)     1
Name: count, dtype: int64


## 1(c)i. Common Time-Domain Features

Based on my reasearch commonly used time-domain features include:

- **Location:** minimum, maximum, mean, median, first and last value.
- **Spread:** standard deviation, variance, range (max - min), first quartile (Q1), third quartile (Q3), interquartile range (IQR), mean absolute deviation.
- **Shape of the distribution:** skewness, kurtosis.
- **Energy / magnitude:** root mean square (RMS), signal energy (sum of squares).
- **Dynamics over time:** autocorrelation, linear trend (slope), zero-crossing rate or mean-crossing rate, number of peaks, mean absolute change between consecutive values.

In this assignment I use the seven features suggested in the problem: minimum, maximum, mean, median, standard deviation, first quartile and third quartile.

## 1(c)ii.


In [4]:
def extract_features(df):
    stats = pd.DataFrame({'min': df.min(), 'max': df.max(), 'mean': df.mean(),
                          'median': df.median(), 'std': df.std(),
                          '1st quart': df.quantile(0.25), '3rd quart': df.quantile(0.75)})
    return pd.Series({f'{stat}{i}': stats.loc[s, stat]
                      for i, s in enumerate(SERIES, start=1) for stat in stats.columns})


features = pd.DataFrame([extract_features(r['data']) for r in records])
features.index = range(1, len(records) + 1)
features.index.name = 'Instance'

print(features.shape)
features.head()

(88, 42)


,min1,max1,mean1,median1,std1,1st quart1,3rd quart1,min2,max2,mean2,...,std5,1st quart5,3rd quart5,min6,max6,mean6,median6,std6,1st quart6,3rd quart6
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


## 1(c)iii. Standard Deviation and 90% Bootstrap Confidence Intervals

For each of the 42 features, I estimate its standard deviation across the 88 files. To quantify the uncertainty of each estimate, I build a 90% bootstrap confidence interval (percentile method):

1. Draw 88 instances with replacement from the original 88 instances.
2. Compute the standard deviation of every feature on this resample.
3. Repeat 1000 times, giving 1000 bootstrap standard deviations per feature.
4. Take the 5th and 95th percentiles of these 1000 values as the lower and upper bounds of the interval.

In [5]:
B = 1000                               # number of bootstrap resamples
rng = np.random.default_rng(0)         # fixed seed for reproducibility
X = features.values                    # 88 x 42 array

# Each resample draws 88 instances with replacement, then recomputes the std of every feature
boot_std = np.array([X[rng.integers(0, len(X), len(X))].std(axis=0, ddof=1)
                     for _ in range(B)])

lower, upper = np.percentile(boot_std, [5, 95], axis=0)   # 90% percentile interval
std_ci = pd.DataFrame({'std': features.std(), 'CI lower (5%)': lower, 'CI upper (95%)': upper})
std_ci.round(3)

,std,CI lower (5%),CI upper (95%)
min1,9.570,8.295,10.819
max1,4.394,3.335,5.273
mean1,5.336,4.706,5.851
median1,5.440,4.841,6.000
std1,1.772,1.573,1.957
1st quart1,6.154,5.615,6.641
3rd quart1,5.139,4.336,5.841
min2,0.000,0.000,0.000
max2,5.063,4.626,5.375
mean2,1.574,1.394,1.702


## 1(c)iv. Selecting the Three Most Important Features


**Selected features: mean, standard deviation and maximum.**

- The mean describes the average signal level, which differs between activities.
- The standard deviation describes how much the signal fluctuates, which also differs between activities and is not redundant with the mean.
- The maximum captures extreme values and complements the other two.
- I do not select the minimum: `min2` and `min4` have a standard deviation of 0 in 1(c)iii (the minimum is 0 for every instance), so they carry no information.

## 2. ISLR 3.7.4

**(a)** The cubic regression should have the lower (or equal) training RSS. The linear model is a special case of the cubic model (set the X² and X³ coefficients to zero), so least squares can always fit the training data at least as well with the cubic model. Even when the true relationship is linear, the extra terms let the cubic model fit some of the random noise, which lowers its training RSS.

**(b)** The linear regression should have the lower test RSS. Since the true relationship is linear, the linear model has no bias and a lower variance. The cubic model overfits: it fits the noise in the training data, and that noise does not carry over to the test data, so its test RSS is higher on average.

**(c)** The cubic regression should have the lower training RSS. The linear model is a special case of the cubic model, so the cubic model can always fit the training data at least as well. This holds no matter how far the true relationship is from linear, and since the true relationship is not linear, the cubic model can also follow the curve more closely.

**(d)** There is not enough information to tell; it depends on how far the true relationship is from linear. If it is close to linear (or the noise is large), the lower variance of the linear model dominates and the linear model has the lower test RSS. If it is far from linear, for example close to cubic, the bias of the linear model is large and the cubic model has the lower test RSS. This is the bias-variance trade-off.